In [0]:
%sql
CREATE TABLE IF NOT EXISTS identifier(:catalog || '.gold.sales_summary') (
    region STRING,
    category STRING,
    sale_year INT,
    sale_month INT,
    total_revenue DECIMAL(14,2),
    total_orders BIGINT,
    prior_year_revenue DECIMAL(14,2),
    yoy_growth_pct DECIMAL(8,2)
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sales_by_region_category_month AS
SELECT
    s.region,
    p.category,
    year(s.sale_date)  AS sale_year,
    month(s.sale_date) AS sale_month,
    SUM(s.sale_amount)  AS total_revenue,
    COUNT(*)             AS total_orders
FROM identifier(:catalog || '.silver.sales_clean') s
JOIN identifier(:catalog || '.silver.customers_clean') c ON s.customer_id = c.id
JOIN identifier(:catalog || '.silver.products_scd2')   p ON s.product_id = p.product_id AND p.is_current = true
JOIN identifier(:catalog || '.silver.suppliers_scd2')  sup ON p.supplier_id = sup.supplier_id AND sup.is_current = true
GROUP BY s.region, p.category, year(s.sale_date), month(s.sale_date);
 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sales_with_yoy AS
SELECT
    region, category, sale_year, sale_month, total_revenue, total_orders,
    LAG(total_revenue) OVER (
        PARTITION BY region, category, sale_month ORDER BY sale_year
    ) AS prior_year_revenue,
    ROUND(
        (total_revenue - LAG(total_revenue) OVER (
            PARTITION BY region, category, sale_month ORDER BY sale_year
        )) / LAG(total_revenue) OVER (
            PARTITION BY region, category, sale_month ORDER BY sale_year
        ) * 100, 2
    ) AS yoy_growth_pct
FROM sales_by_region_category_month;

In [0]:
%sql
INSERT OVERWRITE identifier(:catalog || '.gold.sales_summary')
SELECT region, category, sale_year, sale_month, total_revenue, total_orders,
       prior_year_revenue, yoy_growth_pct
FROM sales_with_yoy;

In [0]:
%sql
SELECT customer_id, customer_name, total_revenue, order_count, revenue_rank
FROM (
    SELECT
        c.id AS customer_id,
        c.name AS customer_name,
        SUM(s.sale_amount) AS total_revenue,
        COUNT(*) AS order_count,
        ROW_NUMBER() OVER (ORDER BY SUM(s.sale_amount) DESC) AS revenue_rank,
        RANK()       OVER (ORDER BY SUM(s.sale_amount) DESC) AS revenue_rank_ties,
        DENSE_RANK() OVER (ORDER BY SUM(s.sale_amount) DESC) AS revenue_dense_rank
    FROM identifier(:catalog || '.silver.sales_clean') s
    JOIN identifier(:catalog || '.silver.customers_clean') c ON s.customer_id = c.id
    GROUP BY c.id, c.name
)
WHERE revenue_rank <= 10
ORDER BY revenue_rank;